In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

In [2]:
df = pd.read_csv('/Users/mukesh/Documents/FLIPKART/data/cleaned_event_dataset.csv')

In [12]:
coords = df[["latitude", "longitude"]].values

# Convert to radians
coords_rad = np.radians(coords)

# Earth radius in km
EARTH_RADIUS = 6371.0088

# eps = 2 km
eps_km = 0.5
eps = eps_km / EARTH_RADIUS

dbscan = DBSCAN(
    eps=eps,
    min_samples=20,
    metric="haversine",
    algorithm="ball_tree"
)

clusters = dbscan.fit_predict(coords_rad)

df["cluster"] = clusters

In [13]:
print(df["cluster"].value_counts())

cluster
 1     2619
-1     1480
 0      830
 2      365
 3      350
 7      346
 4      309
 18     214
 11     184
 6      142
 5      136
 22     129
 14      77
 26      72
 21      65
 17      63
 19      59
 10      52
 24      50
 8       49
 12      49
 27      41
 15      39
 29      35
 28      30
 20      30
 25      29
 30      28
 9       27
 31      26
 32      26
 33      24
 37      24
 16      23
 41      21
 35      21
 23      20
 38      20
 40      18
 36      18
 39      15
 13      14
 34       4
Name: count, dtype: int64


In [14]:
# the Zone statistics
zone_stats = (
    df.groupby("cluster")
    .agg(
        incidents=("id", "count"),
        high_priority=(
            "priority",
            lambda x: (x == "High").sum()
        ),
        road_closures=(
            "requires_road_closure",
            "sum"
        )
    )
)

zone_stats["high_priority_ratio"] = (
    zone_stats["high_priority"]
    / zone_stats["incidents"]
)

print(zone_stats.sort_values(
    "incidents",
    ascending=False
))

         incidents  high_priority  road_closures  high_priority_ratio
cluster                                                              
 1            2619           1860            246             0.710195
-1            1480            350            167             0.236486
 0             830            656             44             0.790361
 2             365            224             26             0.613699
 3             350            315             13             0.900000
 7             346            313             19             0.904624
 4             309            276             17             0.893204
 18            214            214             19             1.000000
 11            184            178              6             0.967391
 6             142              0             17             0.000000
 5             136            120              6             0.882353
 22            129              0             10             0.000000
 14             77  

In [15]:
priority_map = {
    "Low": 1,
    "High": 2
}

event_weights = {
    "vehicle_breakdown":1,
    "pot_holes":2,
    "construction":3,
    "water_logging":3,
    "accident":4,
    "tree_fall":4,
    "public_event":5,
    "procession":5,
    "vip_movement":6,
    "protest":6
}

closure_map = {
    True:3,
    False:0
}

df["severity_score"] = (
    df["priority"].map(priority_map)
    + df["event_cause"].map(event_weights).fillna(1)
    + df["requires_road_closure"].map(closure_map)
)

In [16]:
zone_severity = (
    df.groupby("cluster")
    .agg(
        avg_severity=("severity_score","mean"),
        incidents=("id","count")
    )
)

print(
    zone_severity.sort_values(
        "avg_severity",
        ascending=False
    )
)

         avg_severity  incidents
cluster                         
 28          4.366667         30
 21          4.307692         65
 20          4.300000         30
 6           4.218310        142
 7           4.167630        346
 27          4.146341         41
 40          4.111111         18
 25          4.068966         29
 10          4.057692         52
 24          4.000000         50
 5           3.801471        136
 18          3.794393        214
 34          3.750000          4
 4           3.731392        309
 41          3.714286         21
 8           3.714286         49
 32          3.615385         26
 3           3.571429        350
-1           3.555405       1480
 19          3.508475         59
 38          3.450000         20
 1           3.448263       2619
 15          3.410256         39
 11          3.396739        184
 36          3.388889         18
 29          3.342857         35
 33          3.333333         24
 2           3.323288        365
 13       

In [17]:
cluster_centers = (
    df[df["cluster"] != -1]
    .groupby("cluster")
    [["latitude","longitude"]]
    .mean()
)

print(cluster_centers)

          latitude  longitude
cluster                      
0        13.033440  77.536398
1        12.982207  77.579733
2        12.997022  77.669117
3        13.096076  77.598304
4        12.919382  77.622240
5        12.903064  77.599744
6        12.919866  77.585823
7        12.952513  77.697056
8        12.921880  77.663170
9        12.982484  77.752082
10       13.017106  77.657415
11       12.987568  77.512872
12       12.961141  77.514843
13       13.006935  77.592606
14       12.915268  77.568719
15       12.926323  77.548994
16       12.839181  77.677436
17       13.019271  77.709588
18       13.038396  77.622240
19       12.961926  77.635350
20       12.854967  77.588792
21       13.045137  77.643619
22       13.061478  77.462169
23       13.074627  77.436819
24       12.999396  77.614853
25       12.906402  77.585074
26       12.940730  77.747072
27       12.998084  77.551436
28       12.927168  77.599682
29       12.955901  77.715350
30       13.020310  77.530775
31       1

In [18]:
cluster_centers.to_csv(
    "cluster_centers.csv"
)

In [19]:
import folium

m = folium.Map(
    location=[12.97,77.59],
    zoom_start=10
)

for cluster, center in cluster_centers.iterrows():

    folium.Marker(
        location=[
            center.latitude,
            center.longitude
        ],
        popup=f"Zone {cluster}"
    ).add_to(m)

m.save("cluster_centers.html")

In [20]:
df.to_csv(
    "traffic_events_clustered.csv",
    index=False
)